# RL Arena — treino DQN na GPU do Colab

Este notebook treina os agentes da versão atual (`alg_dqn` ou
`alg_doubledqn_per_dueling`) no Colab. O pacote `rl-arena-colab.zip` contém o
código (e uma cópia deste notebook).

**Passo a passo**

1. No menu, escolha *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.
2. Rode a célula de verificação da GPU.
3. Faça upload do `rl-arena-colab.zip` quando a célula pedir.
4. Rode as células na ordem: extrair, instalar, configurar, treinar, avaliar e baixar.

O treino grava `checkpoints/` (`best.pt`, `latest.pt` e snapshots em `budget/`) e
`logs/` (CSV do treino, `stats.csv` opcional e TensorBoard em `logs/runs`).
A última célula gera `rl-arena-resultados.zip` para download.

**Sobre usar a GPU:** a rede é pequena (3 convoluções + MLP), então o gargalo
continua sendo a simulação do ambiente, que roda em CPU. A GPU tende a ajudar
quando o modo vetorizado (`--envs` > 1, com vários workers em CPU) alimenta
lotes maiores. Se quiser comparar, rode também com `ENVS=8` e sem TensorBoard.

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi -L

## 1. Upload do pacote

Selecione o arquivo `rl-arena-colab.zip` gerado por `bash colab/build_zip.sh`
(na raiz do repositório, o zip fica em `dist/rl-arena-colab.zip`).

In [ ]:
from google.colab import files

print("Selecione o arquivo rl-arena-colab.zip")
enviados = files.upload()
nome = next((n for n in enviados if n.endswith(".zip")), None)
assert nome, "Nenhum .zip foi enviado."
if nome != "rl-arena-colab.zip":
    import os
    os.replace(nome, "rl-arena-colab.zip")
print("Pacote recebido:", nome)

## 2. Extrair e entrar no projeto

In [ ]:
!rm -rf /content/rl-arena
!unzip -q -o rl-arena-colab.zip -d /content
%cd /content/rl-arena
!ls

## 3. Dependências

O PyTorch com CUDA já vem na imagem do Colab. O `requirements-colab.txt`
instala apenas o que falta para treinar e avaliar (sem pygame/render).

In [ ]:
!pip install -q -r colab/requirements-colab.txt
import gymnasium
print("gymnasium:", gymnasium.__version__)

## 4. Configuração do treino

Ajuste os valores e rode a célula seguinte. `ALGORITHM` é o sufixo do pacote
`alg_*` (`dqn` ou `doubledqn_per_dueling`). O adversário `auto` só funciona
com `ENVS = 1`; com `ENVS > 1` use `random`, `rule` ou `coins` e desligue o
TensorBoard (o currículo e a telemetria são por processo).

Flags disponíveis: `--alg`, `--episodes`, `--seed`, `--opponent`,
`--frame-stack`, `--frame-skip`, `--lr`, `--batch-size`, `--envs`,
`--checkpoint-dir`, `--resume`, `--save-every`, `--stats`, `--tensorboard`.

In [ ]:
EPISODES = 300        # episódios adicionais
SEED = 7
ALGORITHM = "dqn"     # "dqn" ou "doubledqn_per_dueling"
OPPONENT = "random"   # "auto" requer ENVS=1 e TENSORBOARD=False
FRAME_STACK = 4
FRAME_SKIP = 3
ENVS = 1              # > 1 liga o treino vetorizado (workers em CPU)
TENSORBOARD = True
CKPT_DIR = "checkpoints/colab"

In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

cmd = [sys.executable, "train.py",
       "--alg", ALGORITHM,
       "--episodes", str(EPISODES),
       "--seed", str(SEED),
       "--opponent", OPPONENT,
       "--frame-stack", str(FRAME_STACK),
       "--frame-skip", str(FRAME_SKIP),
       "--checkpoint-dir", CKPT_DIR]
if ENVS > 1:
    assert OPPONENT != "auto", "--envs > 1 não aceita --opponent auto."
    assert not TENSORBOARD, "--envs > 1 não aceita --tensorboard (nem --stats)."
    cmd += ["--envs", str(ENVS)]
if TENSORBOARD:
    cmd.append("--tensorboard")

Path("logs").mkdir(exist_ok=True)
log_file = open("logs/train-colab.log", "w")
print("Rodando:", shlex.join(cmd))
processo_treino = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
print("Treino iniciado em background (PID", processo_treino.pid, ").")
print("Acompanhe com: !tail -f logs/train-colab.log")

## 5. Acompanhar o treino (log e TensorBoard)

O treino roda em background e a célula anterior retorna na hora. Para ver o
log ao vivo: `!tail -f logs/train-colab.log`. Rode a célula do TensorBoard
enquanto ele roda — ou depois, para ver as curvas já gravadas em
`logs/runs`. A célula seguinte espera o processo terminar antes da avaliação.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/runs

In [ ]:
# Espera o treino em background terminar antes de avaliar.
# Se você reiniciou o kernel, rode a célula de treino de novo (ou confira
# `!tail -n 20 logs/train-colab.log` antes de continuar).
processo_treino.wait()
print("Treino terminou com código", processo_treino.returncode)
!tail -n 20 logs/train-colab.log

## 6. Avaliar o agente (política gulosa)

Avalia o `best.pt` e cada snapshot de orçamento (`budget/ep050.pt`,
`budget/ep100.pt`, ...) contra o adversário `rule`, no protocolo do A/B:
20 episódios com seed 1000. O `evaluate.py` aplica automaticamente o algoritmo
e o frame stack/skip gravados no checkpoint.

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

snapshots = sorted(glob.glob(f"{CKPT_DIR}/budget/ep*.pt"))
melhor = f"{CKPT_DIR}/best.pt"
alvos = ([melhor] if os.path.exists(melhor) else []) + snapshots

for caminho in alvos:
    print("==", caminho)
    subprocess.run([sys.executable, "evaluate.py", "--checkpoint", caminho,
                    "--episodes", "20", "--seed", "1000"], check=True)
    if os.path.exists("logs/evaluation.csv"):
        os.replace("logs/evaluation.csv", f"logs/evaluation-{Path(caminho).stem}.csv")

## 7. Curva de treino

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("logs/training.csv")
df["reward_roll"] = df["reward"].rolling(20, min_periods=1).mean()

plt.figure(figsize=(10, 4))
plt.plot(df["episode"], df["reward"], alpha=0.3, label="recompensa por episódio")
plt.plot(df["episode"], df["reward_roll"], label="média móvel (20)")
plt.xlabel("episódio")
plt.ylabel("recompensa")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Baixar os resultados

Gera `rl-arena-resultados.zip` com `checkpoints/` e `logs/` (inclui os CSVs
de avaliação e os eventos do TensorBoard).

In [ ]:
!zip -qr /content/rl-arena-resultados.zip checkpoints logs
from google.colab import files
files.download("/content/rl-arena-resultados.zip")

## 9. Opcional: retomar o treino

Checkpoints são pesos-only (pesos, otimizador, contadores, algoritmo e config).
A retomada continua a numeração dos episódios, mas buffer e RNGs recomeçam —
não é bit a bit. Use o mesmo `--alg` do checkpoint: o algoritmo é validado
antes de carregar os pesos.

```python
# !python train.py --episodes 100 --seed 7 --alg dqn \
#     --resume checkpoints/colab/latest.pt \
#     --checkpoint-dir checkpoints/colab --tensorboard
```

## 10. Opcional: comparar os algoritmos

A célula abaixo monta comandos para comparar `alg_dqn` e
`alg_doubledqn_per_dueling` com dois lrs. Descomente `subprocess.run` para
rodar uma combinação por vez e compare os snapshots de orçamento com
`evaluate.py`.

In [ ]:
import shlex
import subprocess
import sys

GRID = [
    {"alg": "dqn",                   "lr": 2.5e-4, "batch_size": 64},
    {"alg": "dqn",                   "lr": 1e-3,   "batch_size": 64},
    {"alg": "doubledqn_per_dueling", "lr": 2.5e-4, "batch_size": 64},
    {"alg": "doubledqn_per_dueling", "lr": 1e-3,   "batch_size": 64},
]

for i, ponto in enumerate(GRID):
    cmd = [sys.executable, "train.py",
           "--alg", ponto["alg"],
           "--episodes", str(EPISODES), "--seed", str(SEED),
           "--opponent", OPPONENT,
           "--frame-stack", str(FRAME_STACK), "--frame-skip", str(FRAME_SKIP),
           "--lr", str(ponto["lr"]),
           "--batch-size", str(ponto["batch_size"]),
           "--checkpoint-dir", f"{CKPT_DIR}/grid{i}"]
    print(shlex.join(cmd))
    # subprocess.run(cmd, check=True)